# Temporal Alignment — News and Liquidity Data using OHLCV

This notebook aligns the cleaned GDELT news dataset with the hourly WTI price and 
liquidity data from Yahoo Finance. Each news article is matched to the closest 
trading hour using timestamp rounding. Articles published outside trading hours 
(nights, weekends) are discarded as no liquidity data is available for those 
periods. The resulting dataset merges article metadata with contemporaneous 
liquidity metrics (log volume, price range, Amihud ratio), and constitutes the 
central modeling dataset for the thesis. It is saved to 
`01_data/processed/gdelt_wti_aligned.csv`.

### General imports

In [19]:
import pandas as pd
import numpy as np

In [20]:
df_price = pd.read_csv("../01_data/raw/price/wti_hourly_raw.csv", 
                       index_col='Datetime', 
                       parse_dates=True)
df_price.index = pd.to_datetime(df_price.index, utc=True)

print(f"Precio: {len(df_price)} registros horarios")
print(f"Rango: {df_price.index.min()} a {df_price.index.max()}")

Precio: 11219 registros horarios
Rango: 2024-03-11 10:00:00+00:00 a 2026-03-11 10:00:00+00:00


### Get processed news sorted by date

For each news, find the price of the closest time in hours.

In [21]:
df_news = pd.read_csv("../01_data/processed/gdelt_wti_with_body.csv", parse_dates=['datetime'])
df_news['datetime'] = pd.to_datetime(df_news['datetime'], utc=True)

price_start = df_price.index.min()
price_end = df_price.index.max()

df_news_filtered = df_news[
    (df_news['datetime'] >= price_start) &
    (df_news['datetime'] <= price_end)
].reset_index(drop=True)

df_news_filtered = df_news_filtered.copy()
df_news_filtered['datetime_hour'] = df_news_filtered['datetime'].dt.round('h')

print(f"Noticias antes del filtro: {len(df_news)}")
print(f"Noticias después del filtro: {len(df_news_filtered)}")
print(f"Descartadas: {len(df_news) - len(df_news_filtered)}")

df_price_reset = df_price.copy()
df_price_reset.index.name = 'datetime_hour'
df_price_reset = df_price_reset.reset_index()

# Calcular features de liquidez
df_price_reset['log_volume'] = np.log(df_price_reset['Volume'] + 1)
df_price_reset['price_range'] = df_price_reset['High'] - df_price_reset['Low']
df_price_reset['log_return'] = np.log(df_price_reset['Close'] / df_price_reset['Close'].shift(1))
df_price_reset['amihud'] = df_price_reset['log_return'].abs() / (df_price_reset['Volume'] + 1)

# Merge
df_aligned = df_news_filtered.merge(
    df_price_reset[['datetime_hour', 'Close', 'Volume', 'log_volume', 'price_range', 'log_return', 'amihud']],
    on='datetime_hour',
    how='left'
)

print(f"Artículos alineados: {len(df_aligned)}")
print(f"Sin precio (fuera de horario): {df_aligned['Volume'].isna().sum()}")
print(df_aligned[['title', 'datetime', 'Close', 'log_volume', 'price_range']].head(5))



Noticias antes del filtro: 3290
Noticias después del filtro: 2934
Descartadas: 356
Artículos alineados: 2934
Sin precio (fuera de horario): 611
                                               title  \
0   Will an Economic Rebound in China Lift Prices ?    
1  Oil sprints higher on Monday with Gaza ceasefi...   
2  Oil up on supply concerns as Russia orders out...   
3  Local Gas Prices Lower than Counties North and...   
4                  Tennessee gas prices jump 9 cents   

                   datetime      Close  log_volume  price_range  
0 2024-03-25 04:15:00+00:00  81.150002    6.415097     0.050003  
1 2024-03-25 15:15:00+00:00  81.970001   10.558803     0.380005  
2 2024-03-25 18:30:00+00:00  82.129997   10.116742     0.470001  
3 2024-03-25 21:00:00+00:00        NaN         NaN          NaN  
4 2024-03-25 21:15:00+00:00        NaN         NaN          NaN  


### Discard dates without Close and Volume 

These dates doesn't have this data since they were published outside of the NYMEX market trading schedule (At night or weekends). For this study, makes more sense to discard them.

In [22]:
df_final = df_aligned.dropna(subset=['Close', 'Volume']).reset_index(drop=True)

print(f"Artículos con precio: {len(df_final)}")
print(f"Rango temporal: {df_final['datetime'].min()} a {df_final['datetime'].max()}")
print(f"\nEstadísticas de liquidez:")
print(df_final[['log_volume', 'price_range', 'amihud']].describe().round(4))

Artículos con precio: 2323
Rango temporal: 2024-03-25 04:15:00+00:00 a 2026-02-20 19:30:00+00:00

Estadísticas de liquidez:
       log_volume  price_range     amihud
count   2323.0000    2323.0000  2323.0000
mean       8.1725       0.3791     0.0001
std        2.4290       0.2761     0.0008
min        0.0000       0.0000     0.0000
25%        7.3476       0.2000     0.0000
50%        8.6618       0.3100     0.0000
75%        9.7374       0.4900     0.0000
max       11.7523       3.4200     0.0191


### Save data

In [23]:
filename = "gdelt_wti_aligned.csv"
df_final.to_csv(f"../01_data/processed/{filename}", index=False)
print(f"\nGuardado en 01_data/processed/{filename}")


Guardado en 01_data/processed/gdelt_wti_aligned.csv
